#### **Zadanie LSTM - generator przepisów z pełnej bazy danych**

In [13]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Hiperparametry
SEQ_LENGTH = 40
BATCH_SIZE = 64
HIDDEN_DIM = 256
NUM_LAYERS = 1
EMBEDDING_DIM = 64
EPOCHS = 10
LEARNING_RATE = 0.002
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Używane urządzenie: {DEVICE}")

Używane urządzenie: cpu


In [21]:
import urllib.request
import json
import numpy as np

print("Pobieranie bazy danych z Github (to zajmie chwilkę)...")
# Słynna baza przepisów kulinarnych Epicurious (20 000+ przepisów)
url = "https://raw.githubusercontent.com/Cyral/Bakeoof/master/full_format_recipes.json"

# Wczytujemy całą odpowiedź i dekodujemy z JSON do listy w Pythonie
with urllib.request.urlopen(url) as response:
    raw_data = response.read().decode('utf-8')
    recipes = json.loads(raw_data)

print("Formatowanie danych do postaci tekstowej...")
TEXT_DATA = ""
count = 0

for recipe in recipes:
    # Ograniczamy do 300 przepisów
    if count >= 300: 
        break
        
    title = recipe.get('title', 'Unknown')
    
    # Składniki to lista, łączymy w jeden tekst po przecinku
    ingredients = recipe.get('ingredients', [])
    if isinstance(ingredients, list):
        ingredients = ", ".join(ingredients)
        
    # Instrukcje to również lista, łączymy spacją
    directions = recipe.get('directions', [])
    if isinstance(directions, list):
        directions = " ".join(directions)
        
    # Sklejamy to do naszego wyuczonego formatu
    recipe_text = f"Title: {title}\nIngredients: {ingredients}\nDirections: {directions}\n[RECIPE_END]\n\n"
    TEXT_DATA += recipe_text
    count += 1

# Tworzenie słownika znaków (Vocabulary) z nowego, gigantycznego tekstu
chars = tuple(set(TEXT_DATA))
int2char = dict(enumerate(chars))
char2int = {ch: ii for ii, ch in int2char.items()}
VOCAB_SIZE = len(chars)

# Zakodowanie całego tekstu jako liczby
encoded_text = np.array([char2int[ch] for ch in TEXT_DATA])

print(f"Gotowe! Sformatowano dokładnie {count} przepisów.")
print(f"Rozmiar wczytanego tekstu: {len(TEXT_DATA) / 1024 / 1024:.2f} MB")
print(f"Vocab Size (rozmiar alfabetu znaków): {VOCAB_SIZE}")
print("-" * 50)
print("Przykładowy wycinek:\n")
print(TEXT_DATA[:600])

Pobieranie bazy danych z Github (to zajmie chwilkę)...
Formatowanie danych do postaci tekstowej...
Gotowe! Sformatowano dokładnie 300 przepisów.
Rozmiar wczytanego tekstu: 0.38 MB
Vocab Size (rozmiar alfabetu znaków): 99
--------------------------------------------------
Przykładowy wycinek:

Title: Lentil, Apple, and Turkey Wrap 
Ingredients: 4 cups low-sodium vegetable or chicken stock, 1 cup dried brown lentils, 1/2 cup dried French green lentils, 2 stalks celery, chopped, 1 large carrot, peeled and chopped, 1 sprig fresh thyme, 1 teaspoon kosher salt, 1 medium tomato, cored, seeded, and diced, 1 small Fuji apple, cored and diced, 1 tablespoon freshly squeezed lemon juice, 2 teaspoons extra-virgin olive oil, Freshly ground black pepper to taste, 3 sheets whole-wheat lavash, cut in half crosswise, or 6 (12-inch) flour tortillas, 3/4 pound turkey breast, thinly sliced, 1/2 head Bi


In [23]:
#przygotowanie danych
class RecipeDataset(Dataset):
    def __init__(self, encoded_text, seq_length):
        self.encoded_text = encoded_text
        self.seq_length = seq_length

    def __len__(self):
        return len(self.encoded_text) - self.seq_length

    def __getitem__(self, idx):
        x = self.encoded_text[idx : idx + self.seq_length]
        y = self.encoded_text[idx + 1 : idx + self.seq_length + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

dataset = RecipeDataset(encoded_text, SEQ_LENGTH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

In [25]:
#Model
class RecipeGeneratorLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        out, hidden = self.lstm(embedded, hidden)
        logits = self.fc(out) 
        return logits, hidden

def generate_text(model, start_str="Title:", length=150, temperature=0.8):
    model.eval()
    chars_input = [char2int.get(c, 0) for c in start_str]
    input_tensor = torch.tensor(chars_input).unsqueeze(0).to(DEVICE)
    
    generated_text = start_str
    hidden = None 
    
    with torch.no_grad():
        for _ in range(length):
            output, hidden = model(input_tensor, hidden)
            logits = output[0, -1, :] / temperature
            probs = torch.softmax(logits, dim=0)
            next_char_idx = torch.multinomial(probs, num_samples=1).item()
            
            generated_text += int2char[next_char_idx]
            input_tensor = torch.tensor([[next_char_idx]]).to(DEVICE)
            
    return generated_text

In [10]:
# Jeśli nie masz tej biblioteki, pierwsza linijka ją zainstaluje
!pip install tqdm -q

In [27]:
import torch
import torch.nn as nn
from tqdm.notebook import tqdm  # Ta wersja działa najlepiej w Jupyter/Colab
from IPython.display import clear_output

# Inicjalizacja modelu i optymalizatora
model = RecipeGeneratorLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Rozpoczynamy trenowanie...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0
    
    # Pasek postępu dla aktualnej epoki (pokazuje ilość przetworzonych paczek)
    progress_bar = tqdm(dataloader, desc=f'Epoka {epoch}/{EPOCHS}', leave=False)
    
    for x_batch, y_batch in progress_bar:
        x_batch, y_batch = x_batch.to(DEVICE), y_batch.to(DEVICE)
        
        optimizer.zero_grad()
        
        # 1. Przepuszczamy sekwencję przez model
        output, _ = model(x_batch)
        
        # 2. Obliczanie błędu
        # Musimy spłaszczyć wymiary! 
        # output ma kształt: [batch_size, seq_length, vocab_size] -> [batch_size * seq_length, vocab_size]
        # y_batch ma kształt: [batch_size, seq_length] -> [batch_size * seq_length]
        loss = criterion(output.view(-1, VOCAB_SIZE), y_batch.view(-1))
        
        # 3. Krok wstecz (Backpropagation)
        loss.backward()
        
        # 4. Zabezpieczenie przed "eksplodującymi gradientami" (częsty problem w LSTM)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        
        # 5. Aktualizacja wag
        optimizer.step()
        epoch_loss += loss.item()
        
        # Wyświetlanie aktualnego błędu na pasku
        progress_bar.set_postfix({'strata': f'{loss.item():.4f}'})
        
    # --- PODSUMOWANIE EPOKI ---
    clear_output(wait=True)
    avg_loss = epoch_loss / len(dataloader)
    print(f"=== Epoka: {epoch}/{EPOCHS} | Średnia strata (Loss): {avg_loss:.4f} ===")
    
    # Używamy angielskiego "Title: ", bo z pliku JSON pobraliśmy angielską bazę.
    sample_text = generate_text(model, start_str="Title: ", length=250, temperature=0.8)
    print(f"Próbka z tej epoki:\n\n{sample_text}\n{'-'*60}")

print("\nKoniec trenowania!")

=== Epoka: 10/10 | Średnia strata (Loss): 0.5566 ===
Próbka z tej epoki:

Title: Dough 
Ingredients: 2 tablespoons whole coriander seeds, 1 teaspoon grated, 3/4 cup chopped fresh basil, tomatoes, peeled, cored, and reserved chopped black oranges, 1 1/2 cups water to a boil. Stir until pouring water.) Just before continuing. Prehe
------------------------------------------------------------

Koniec trenowania!


In [29]:
import torch

def generate_text(model, start_str="Title: ", length=400, temperature=0.8):
    # Przełączamy model w tryb testowy (wyłącza np. warstwy Dropout)
    model.eval()
    
    # Inicjalizacja stanu ukrytego - zaczynamy z czystą kartą
    hidden = None 
    
    text_generated = start_str
    
    # Przekształcamy tekst startowy na listę indeksów (liczb)
    input_indices = []
    for ch in start_str:
        # Zabezpieczenie: jeśli wpiszemy znak, którego nie ma w słowniku, zamień go na spację
        if ch in char2int:
            input_indices.append(char2int[ch])
        else:
            input_indices.append(char2int.get(' ', 0))
            
    # Zmieniamy w tensor o kształcie [batch_size=1, seq_length] i wysyłamy na GPU/CPU
    input_tensor = torch.tensor(input_indices).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        for i in range(length):
            # Uwaga: podajemy do modelu input_tensor oraz poprzedni stan 'hidden'
            output, hidden = model(input_tensor, hidden)
            
            # Interesuje nas tylko przewidywanie dla OSTATNIEGO znaku w sekwencji
            # output ma kształt: [batch=1, seq_len, vocab_size] -> bierzemy [-1] z seq_len
            last_logits = output[0, -1, :] 
            
            # --- TEMPERATURA ---
            last_logits = last_logits / temperature
            probs = torch.softmax(last_logits, dim=0)
            
            # Losujemy następny znak z użyciem prawdopodobieństw 
            # (zamiast wybierać zawsze ten najwyższy przez argmax)
            next_index = torch.multinomial(probs, num_samples=1).item()
            next_char = int2char[next_index]
            
            text_generated += next_char
            
            # Przewidziany znak staje się JEDYNYM wejściem w kolejnym kroku pętli
            input_tensor = torch.tensor([[next_index]]).to(DEVICE)
            
            # Jeśli model "zrozumiał", że skończył przepis, przerywamy pętlę
            if "[RECIPE_END]" in text_generated:
                break
                
    return text_generated

# ==========================================
# EKSPERYMENTY STUDENCKIE
# ==========================================
print("Kucharz AI jest gotowy do pracy!\n")

print("--- EKSPERYMENT 1: Bezpiecznie (Niska temperatura = 0.3) ---")
# Model wybiera najbardziej prawdopodobne znaki. Tekst będzie poprawny, ale może być nudny.
print(generate_text(model, start_str="Title: Chocolate Cake\nIngredients:", length=400, temperature=0.3))
print("\n" + "="*50 + "\n")

print("--- EKSPERYMENT 2: Kulinarna improwizacja (Zbalansowana temperatura = 0.8) ---")
# Idealny balans między sensem a nowymi wymysłami
print(generate_text(model, start_str="Title: Spicy Chicken\nIngredients:", length=400, temperature=0.8))
print("\n" + "="*50 + "\n")

print("--- EKSPERYMENT 3: Szaleństwo (Wysoka temperatura = 1.3) ---")
# Model zaczyna mocno losować. Prawdopodobnie zgubi gramatykę i wymyśli nowe, nieistniejące słowa.
print(generate_text(model, start_str="Title: Magic Soup\nIngredients:", length=400, temperature=1.3))
print("\n" + "="*50 + "\n")

Kucharz AI jest gotowy do pracy!

--- EKSPERYMENT 1: Bezpiecznie (Niska temperatura = 0.3) ---
Title: Chocolate Cake
Ingredients: 3 tablespoons olive oil, 1 tablespoon minced peeled fresh ginger, 1 teaspoon finely grated lemon zest, 1 tablespoon chopped fresh thyme, 1 teaspoon salt, 1/2 teaspoon ground cinnamon, 1 1/2 teaspoons salt, 1/2 teaspoon salt, 1/2 teaspoon salt, 1/2 teaspoon salt, 1/2 teaspoon salt, 1/2 teaspoon ground cinnamon, 1 1/2 teaspoons dried thyme, 1/4 teaspoon salt, 1/2 teaspoon dried thyme, 1/4 teaspoon 


--- EKSPERYMENT 2: Kulinarna improwizacja (Zbalansowana temperatura = 0.8) ---
Title: Spicy Chicken
Ingredients: 1/4 cup coarsely chopped walnuts, 1/2 teaspoon salt, 3 large egg yolks, 1/2 cup vegetable oil, 1 large egg yolk, 3 teaspoons olive oil in heavy large skillet over medium-high heat until hot but the roasting pan. Cook, turning water, 2 tablespoons puret over medium heat. Add garlic and sauté 2 minutes. Add the beef ins trimmed, 2 tablespoons kosher salt,